In [ ]:
import pypsa

import matplotlib.pyplot as plt

In [ ]:
from _helpers_notebooks import mock_snakemake

snakemake = mock_snakemake(
    "plot_compare_lcox",
    scenario="default",  # default, penalty-sa, penalty-ea, penalty-nwa, penalty-oc
)

In [ ]:
config = snakemake.config

In [ ]:
colors = config["colors"]

### Read networks in Network collection

In [ ]:
scenarios = snakemake.input.keys()

In [ ]:
nc = {}

for scenario in scenarios:
    model_fn = snakemake.input[scenario]

    n = pypsa.Network(model_fn)
    n.name = scenario
    nc[scenario] = n

nc = pypsa.NetworkCollection(list(nc.values()))

### Costs

In [ ]:
hbi_demand = nc.statistics.withdrawal().to_frame().loc["Load", :, "hbi_demand"]
hbi_demand

In [ ]:
df = nc.statistics.system_cost(groupby="carrier").to_frame("systemcost")
df

In [ ]:
fom = nc.statistics.fom(groupby="carrier").to_frame("fom")

In [ ]:
df["systemcost+fom"] = df["systemcost"].add(fom["fom"], fill_value=0)

In [ ]:
df["systemcost+fom_per_hbi"] = df["systemcost+fom"].div(hbi_demand.squeeze().rename_axis("network"), level="network")
df

### Plot

In [ ]:
# Pivot to get networks as rows and carriers as columns
plot_df = df.pivot_table(
    index="network", columns="carrier", values="systemcost+fom_per_hbi", aggfunc="sum"
)

plot_df.rename(
    columns={
        "renewable_solar": "solar",
        "renewable_onwind": "onshore wind",
        "hydrogen": "hydrogen storage",
        "electrolysis": "electrolysis",
        "direct_reduction_furnace": "direct reduction furnace",
    },
    inplace=True,
)

order = ["solar", "onshore wind", "hydrogen storage", "electrolysis", "direct reduction furnace"]

plot_df = plot_df[order]

In [ ]:
# Create plot
ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 4),
    color=[config["colors"][c] for c in order],
    alpha=0.8,
)

# ---- Axis labels ----
ax.set_ylabel("Total cost in €/t$_{hbi}$")
ax.set_xlabel("Scenario")

# ---- Title ----
# ax.set_title("Cost of Steel")

# ---- Y-axis grid ----
ax.yaxis.grid(True, linestyle="-", alpha=0.6)
ax.set_axisbelow(True)

# ---- Add total values on top of bars ----
totals = plot_df.sum(axis=1)

for i, total in enumerate(totals):
    ax.text(
        i,
        total,
        f"{total:.1f}",  # adjust scaling if needed
        ha="center",
        va="bottom",
        fontsize=10,
    )

# ---- Flip legend order ----
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[::-1],
    labels[::-1],
    title="Cost component",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(snakemake.output.lcox_comparison, dpi=300)
plt.savefig(snakemake.output.lcox_comparison_png, dpi=300)
plt.show()

<!-- ### Statistics clarifiction (to be deleted) -->

In [ ]:
# system_cost = nc["eu_1"].statistics.system_cost().div(1e6).sum()
# system_cost.round(2)

In [ ]:
# expanded_capex = nc["eu_1"].statistics.expanded_capex().div(1e6).sum()
# expanded_capex.round(2)

In [ ]:
# opex = nc["eu_1"].statistics.opex().div(1e6).sum()
# opex.round(2)

<!-- -> system_cost = expanded_capex + opex -->

In [ ]:
# objective = nc["eu_1"].objective / 1e6
# objective.round(2)

In [ ]:
# fom = nc["eu_1"].statistics.fom().div(1e6).sum()
# fom.round(2)

<!-- -> objective = system_cost + fom = expanded_capex + opex + fom -->

In [ ]:
# nc["eu_1"].statistics.overnight_cost().div(1e6).sum()